# File Formats, Storage and Lakehouse Table Formats

Notebook for **Chapter 3** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 Text Formats: CSV and JSON, and Their Traps

In [ ]:
import csv, io, json
tricky = [("1","Doe, John",'said "hi"',"007"), ("2","line\nbreak","x","1.50")]
b = io.StringIO(); csv.writer(b).writerows(tricky); print(repr(b.getvalue()))
naive = b.getvalue().split("\n"); print("naive split gives", len(naive)-1, "lines for 2 records")
print("csv module read:", list(csv.reader(io.StringIO(b.getvalue()))))
print("leading zero lost if cast to int:", int("007"))
print("json float:", 0.1+0.2, "| big int:", json.loads('{"x": 12345678901234567890}')["x"])

## 3.2 Columnar Formats: Parquet, ORC, Avro, Arrow

In [ ]:
import csv, io, zlib, random
random.seed(7)
N = 20000
cities = ['Pune','Mumbai','Delhi','Chennai','Kolkata']
rows = [(i, random.choice(cities), random.choice(['A','B','C']),
         round(random.uniform(10,500),2), f"2024-01-{random.randint(1,28):02d}") for i in range(N)]
csv_buf = io.StringIO(); csv.writer(csv_buf).writerows(rows); row_bytes = csv_buf.getvalue().encode()
cols = list(zip(*rows))
col_bytes = b"".join(("\n".join(map(str, c)) + "\n").encode() for c in cols)
print("raw row-layout bytes   :", len(row_bytes))
print("raw column-layout bytes:", len(col_bytes))
print("zlib row-layout        :", len(zlib.compress(row_bytes, 6)))
print("zlib column-layout     :", len(zlib.compress(col_bytes, 6)))
for n, c in zip(['id','city','tier','amount','day'], cols):
    b = ("\n".join(map(str, c)) + "\n").encode()
    print(f"  col {n:7s} raw {len(b):7d} zlib {len(zlib.compress(b,6)):7d}")

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq
table = pa.Table.from_pylist([{"id": 1, "city": "Pune", "amount": 250.0}])
pq.write_table(table, "orders.parquet", compression="zstd")
print(pq.ParquetFile("orders.parquet").metadata)        # row groups, rows, footer info
only_amount = pq.read_table("orders.parquet", columns=["amount"])   # column pruning
filtered = pq.read_table("orders.parquet", filters=[("amount", ">", 100)])  # predicate pushdown

## 3.3 Compression, Partitioning and the Small-File Problem

In [ ]:
import os, tempfile, time, random
random.seed(7)
N = 20000
rows = [(i, random.choice(['Pune','Mumbai','Delhi']), random.randint(1,28)) for i in range(N)]
# partition by day
base = tempfile.mkdtemp()
for i, city, d in rows:
    p = f"{base}/day=2024-01-{d:02d}"; os.makedirs(p, exist_ok=True)
    with open(p + "/part.csv", "a") as f: f.write(f"{i},{city},{d}\n")
parts = sorted(os.listdir(base)); print("partitions:", len(parts), parts[:3])
n = sum(1 for _ in open(f"{base}/day=2024-01-05/part.csv"))
print(f"query day=2024-01-05 reads 1 of {len(parts)} folders, {n} of {N} rows")
# small files: same lines, 1 file vs 2000 files
lines = [f"{i},{c},{d}\n" for i, c, d in rows]
d1 = tempfile.mkdtemp(); open(d1 + "/one.csv", "w").writelines(lines)
d2 = tempfile.mkdtemp()
for k in range(2000): open(f"{d2}/p{k}.csv", "w").writelines(lines[k*10:(k+1)*10])
def readall(d):
    t = time.perf_counter(); n = 0
    for fn in os.listdir(d):
        with open(os.path.join(d, fn)) as f: n += sum(1 for _ in f)
    return n, time.perf_counter() - t
for lab, d in (("1 file", d1), ("2000 files", d2)):
    n, t = readall(d); print(f"{lab:11s} lines={n} read_time={t*1000:.1f} ms")

## 3.4 Table Formats: Delta Lake, Iceberg, Hudi

In [ ]:
import json, os, tempfile
root = tempfile.mkdtemp(); os.makedirs(root + "/_log")
def write_file(name, rows): json.dump(rows, open(f"{root}/{name}", "w"))
def commit(version, actions): json.dump(actions, open(f"{root}/_log/{version:03d}.json", "w"))
def snapshot(as_of=None):
    files = set()
    for fn in sorted(os.listdir(root + "/_log")):
        v = int(fn[:3])
        if as_of is not None and v > as_of: break
        for a in json.load(open(f"{root}/_log/{fn}")):
            (files.add if a["op"] == "add" else files.discard)(a["file"])
    rows = []
    for f in sorted(files): rows += json.load(open(f"{root}/{f}"))
    return sorted(files), rows
write_file("f1.json", [{"id":1,"city":"Pune"}, {"id":2,"city":"Delhi"}]); commit(0, [{"op":"add","file":"f1.json"}])
write_file("f2.json", [{"id":3,"city":"Goa"}]);                           commit(1, [{"op":"add","file":"f2.json"}])
# UPDATE id=2 -> Mumbai: never edit a file. Rewrite f1 without id 2, write f3, commit both together.
write_file("f1b.json", [{"id":1,"city":"Pune"}])
write_file("f3.json",  [{"id":2,"city":"Mumbai"}])
commit(2, [{"op":"remove","file":"f1.json"}, {"op":"add","file":"f1b.json"}, {"op":"add","file":"f3.json"}])
for v in (0, 1, 2, None):
    files, rows = snapshot(v); print("version", v, "files", files, "rows", [(r['id'], r['city']) for r in rows])

## 3.5 Table Format Internals: Snapshots, Conflicts, Z-order and Deletes

Section 3.4 told you what Delta Lake, Iceberg and Hudi give you: transactions, time travel and schema evolution on top of plain Parquet files. This section shows *how*, with toy models you can run. Once you see the mechanism, the features stop being magic and the interview questions (why can two writers conflict, why does a delete not free disk space, what does Z-order do) become easy.

In [ ]:
# A tiny table format: snapshots point to manifests, manifests list data files.
# Commits are atomic because the only mutable thing is one pointer: "current snapshot id".
class Table:
    def __init__(self):
        self.snapshots = {0: []}        # snapshot id -> list of data files (via the manifest)
        self.current = 0
    def files(self, snap=None):
        return self.snapshots[self.current if snap is None else snap]
    def commit(self, base, add=(), remove=()):
        """Optimistic concurrency: succeed only if nobody committed since 'base'."""
        if base != self.current:
            return False
        new = [f for f in self.snapshots[base] if f not in remove] + list(add)
        sid = max(self.snapshots) + 1
        self.snapshots[sid] = new
        self.current = sid               # the single atomic step
        return True

t = Table()
print("commit load1:", t.commit(0, add=["f1.parquet", "f2.parquet"]), "-> snapshot", t.current)
# Two writers both start from snapshot 1.
base_a = base_b = t.current
print("writer A appends f3 :", t.commit(base_a, add=["f3.parquet"]), "-> snapshot", t.current)
print("writer B compacts f1+f2 into f12:", t.commit(base_b, add=["f12.parquet"], remove=["f1.parquet", "f2.parquet"]))
# B must retry: re-read the current state, redo the compaction on top of it.
base_b = t.current
print("writer B retry      :", t.commit(base_b, add=["f12.parquet"], remove=["f1.parquet", "f2.parquet"]), "-> snapshot", t.current)
print("current files      :", t.files())
print("time travel, snap 1:", t.files(1))
print("reader that started on snapshot 2 still sees:", t.files(2))

commit load1: True -> snapshot 1
writer A appends f3 : True -> snapshot 2
writer B compacts f1+f2 into f12: False
writer B retry      : True -> snapshot 3
current files      : ['f3.parquet', 'f12.parquet']
time travel, snap 1: ['f1.parquet', 'f2.parquet']
reader that started on snapshot 2 still sees: ['f1.parquet', 'f2.parquet', 'f3.parquet']


In [ ]:
# Z-order: interleave the bits of two columns so rows close in BOTH columns sit close in the file.
import random
random.seed(3)
N, FILE = 4096, 64                           # 64 files of 64 rows
rows = [(random.randrange(64), random.randrange(64)) for _ in range(N)]   # (x, y), both 0..63
def zkey(x, y):
    z = 0
    for b in range(6):
        z |= ((x >> b) & 1) << (2*b) | ((y >> b) & 1) << (2*b + 1)
    return z
def files_read(order, qx, qy):
    n = 0
    for i in range(0, N, FILE):
        blk = order[i:i+FILE]
        xs = [r[0] for r in blk]; ys = [r[1] for r in blk]
        okx = qx is None or (min(xs) <= qx[1] and max(xs) >= qx[0])
        oky = qy is None or (min(ys) <= qy[1] and max(ys) >= qy[0])
        if okx and oky: n += 1
    return n
by_x = sorted(rows)                                    # sorted by x, then y
by_z = sorted(rows, key=lambda r: zkey(*r))
print("filter x in 8..15 only    : sort by x ->", files_read(by_x, (8, 15), None), "files; z-order ->", files_read(by_z, (8, 15), None))
print("filter y in 8..15 only    : sort by x ->", files_read(by_x, None, (8, 15)), "files; z-order ->", files_read(by_z, None, (8, 15)))
print("filter both 8..15         : sort by x ->", files_read(by_x, (8, 15), (8, 15)), "files; z-order ->", files_read(by_z, (8, 15), (8, 15)))
print("total files:", N // FILE)

filter x in 8..15 only    : sort by x -> 9 files; z-order -> 18
filter y in 8..15 only    : sort by x -> 64 files; z-order -> 12
filter both 8..15         : sort by x -> 9 files; z-order -> 3
total files: 64


In [ ]:
# Deletes in an immutable-file world: rewrite the file (copy on write) or mark rows (deletion vector).
rows = [{"id": i, "email": f"user{i}@x.com"} for i in range(1, 1_000_001, 1000)]  # 1,000 sample rows
FILE_ROWS = 100
files = [rows[i:i+FILE_ROWS] for i in range(0, len(rows), FILE_ROWS)]
victim = 501001                      # the id to erase (right to erasure request)
holder = [i for i, f in enumerate(files) if any(r["id"] == victim for r in f)]
print("files in table:", len(files), "| file holding the row:", holder)
cow_rows_written = sum(len(files[i]) - 1 for i in holder)
print("copy-on-write: rewrite that file, rows written =", cow_rows_written)
dv = {holder[0]: {next(j for j, r in enumerate(files[holder[0]]) if r["id"] == victim)}}
print("deletion vector: rows written = 0, bitmap stored:", dv)
live = [r for i, f in enumerate(files) for j, r in enumerate(f) if j not in dv.get(i, set())]
print("rows visible to readers:", len(live), "(was", len(rows), ")")
print("but the old bytes still exist on disk until compaction and vacuum remove the old file version")

files in table: 10 | file holding the row: [5]
copy-on-write: rewrite that file, rows written = 99
deletion vector: rows written = 0, bitmap stored: {5: {1}}
rows visible to readers: 999 (was 1000 )
but the old bytes still exist on disk until compaction and vacuum remove the old file version


## Chapter recap

## Practice set

**1.** (easy) Which format is human readable and has no types: Parquet, CSV, Avro? **Answer:** CSV.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch03.html